In [3]:
# Inspect BIS file structure (raw lines + robust load)

from pathlib import Path
import pandas as pd

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
BIS_FILE = PROJECT / "data" / "raw" / "bis_credit" / "bis_dp_search_export_20260924-165351.csv"

print("=" * 60)
print("BIS Credit – Raw Structure Check")
print("=" * 60)
print(f"File: {BIS_FILE}")
print(f"Size: {BIS_FILE.stat().st_size / 1e6:.2f} MB")

# 1) Show first 15 raw lines
print("\n--- First 15 raw lines ---")
with open(BIS_FILE, "r", encoding="utf-8", errors="replace") as f:
    for i, line in enumerate(f):
        if i >= 15:
            break
        print(f"{i:02d}| {line.rstrip()[:200]}")

# 2) Try common skip patterns
print("\n--- Trying load strategies ---")
for skip in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    try:
        df = pd.read_csv(BIS_FILE, skiprows=skip, low_memory=False, encoding="utf-8")
        print(f"skiprows={skip}: shape={df.shape} | cols={list(df.columns)[:6]}...")
        if df.shape[1] > 5 and df.shape[0] > 10:
            print("  --> candidate OK")
            print(df.head(2).to_string())
            break
    except Exception as e:
        print(f"skiprows={skip}: FAILED ({type(e).__name__}: {str(e)[:80]})")

print("\nCell completed.")

BIS Credit – Raw Structure Check
File: D:\univercity\omid_project\economics_project\data\raw\bis_credit\bis_dp_search_export_20260924-165351.csv
Size: 0.08 MB

--- First 15 raw lines ---
00| ﻿Time Series Search Export,Search term,Search filters,,Last update of Credit to the non-financial sector data,Downloaded at,Source,Source URL,Download URL,About Credit to the non-financial sector
01| ,,TIMESPAN:Timespan,1980-01-01_2026-03-31:,2026-09-14,2026-09-24 18:53 CEST,BIS,https://data.bis.org/topics/TOTAL_CREDIT/data?data_view=timeseries&page_size=100&filter=TC_BORROWERS%3DG%257CP%255EUNIT_
02| 
03| KEY:Timeseries Key,BIS:WS_TC(2.0):Q.AU.G.A.M.770.A,BIS:WS_TC(2.0):Q.AU.G.A.N.770.A,BIS:WS_TC(2.0):Q.AU.P.A.M.770.A,BIS:WS_TC(2.0):Q.AU.P.B.M.770.A,BIS:WS_TC(2.0):Q.BE.G.A.M.770.A,BIS:WS_TC(2.0):Q.BE.G
04| FREQ:Frequency,Q:Quarterly,Q:Quarterly,Q:Quarterly,Q:Quarterly,Q:Quarterly,Q:Quarterly,Q:Quarterly,Q:Quarterly,Q:Quarterly,Q:Quarterly,Q:Quarterly,Q:Quarterly,Q:Quarterly,Q:Quarterly,Q:Quarterly

In [4]:
# Parse BIS Total Credit wide export correctly

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
BIS_FILE = PROJECT / "data" / "raw" / "bis_credit" / "bis_dp_search_export_20260924-165351.csv"
OUT_DIR = PROJECT / "data" / "raw" / "bis_credit"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("BIS Credit – Proper Parse")
print("=" * 60)

# Read all lines
with open(BIS_FILE, "r", encoding="utf-8-sig", errors="replace") as f:
    lines = f.readlines()

print(f"Total lines: {len(lines)}")

# Metadata rows (3..12) contain series definitions
# Row 3 = KEY
# Row 5 = BORROWERS_CTY
# Row 6 = TC_BORROWERS
# Row 7 = TC_LENDERS
# Row 8 = VALUATION
# Row 9 = UNIT_TYPE
# Row 13 = TIME_PERIOD header
# Row 14+ = data

def split_csv_line(line):
    return [x.strip() for x in line.rstrip("\n").split(",")]

key_row = split_csv_line(lines[3])
cty_row = split_csv_line(lines[5])
bor_row = split_csv_line(lines[6])
lend_row = split_csv_line(lines[7])
val_row = split_csv_line(lines[8])
unit_row = split_csv_line(lines[9])

# First column is label; series start from index 1
n_series = len(key_row) - 1
print(f"Detected series columns: {n_series}")

# Build series metadata
meta = []
for i in range(1, len(key_row)):
    meta.append({
        "col_idx": i,
        "key": key_row[i] if i < len(key_row) else "",
        "country": cty_row[i].split(":")[0] if i < len(cty_row) and ":" in cty_row[i] else (cty_row[i] if i < len(cty_row) else ""),
        "country_label": cty_row[i].split(":")[-1] if i < len(cty_row) and ":" in cty_row[i] else "",
        "borrowers": bor_row[i].split(":")[0] if i < len(bor_row) and ":" in bor_row[i] else (bor_row[i] if i < len(bor_row) else ""),
        "lenders": lend_row[i].split(":")[0] if i < len(lend_row) and ":" in lend_row[i] else (lend_row[i] if i < len(lend_row) else ""),
        "valuation": val_row[i].split(":")[0] if i < len(val_row) and ":" in val_row[i] else (val_row[i] if i < len(val_row) else ""),
        "unit": unit_row[i].split(":")[0] if i < len(unit_row) and ":" in unit_row[i] else (unit_row[i] if i < len(unit_row) else ""),
    })

meta_df = pd.DataFrame(meta)
print("\nSeries metadata sample:")
print(meta_df.head(8).to_string(index=False))
print(f"\nCountries in file: {sorted(meta_df['country'].unique())}")

# Data rows start at line 14 (index 14)
data_rows = []
for line in lines[14:]:
    parts = split_csv_line(line)
    if not parts or parts[0] == "":
        continue
    period = parts[0]
    values = parts[1:]
    data_rows.append([period] + values)

# Align columns
max_cols = max(len(r) for r in data_rows)
cols = ["TIME_PERIOD"] + [f"s{i}" for i in range(1, max_cols)]
wide = pd.DataFrame(data_rows, columns=cols[:max_cols])

print(f"\nWide data shape: {wide.shape}")

# Melt to long
long_records = []
for _, m in meta_df.iterrows():
    col_name = f"s{m['col_idx']}"
    if col_name not in wide.columns:
        continue
    tmp = wide[["TIME_PERIOD", col_name]].copy()
    tmp = tmp.rename(columns={col_name: "value"})
    tmp["value"] = pd.to_numeric(tmp["value"], errors="coerce")
    tmp["country"] = m["country"]
    tmp["country_label"] = m["country_label"]
    tmp["borrowers"] = m["borrowers"]
    tmp["lenders"] = m["lenders"]
    tmp["valuation"] = m["valuation"]
    tmp["unit"] = m["unit"]
    tmp["key"] = m["key"]
    long_records.append(tmp)

bis_long = pd.concat(long_records, ignore_index=True)
bis_long = bis_long.dropna(subset=["value"])

# Parse year
bis_long["year"] = pd.to_datetime(bis_long["TIME_PERIOD"], errors="coerce").dt.year

print(f"\nLong format shape: {bis_long.shape}")
print(f"Years: {bis_long['year'].min()} → {bis_long['year'].max()}")
print(f"Countries: {sorted(bis_long['country'].dropna().unique())}")
print(f"Borrowers sectors: {bis_long['borrowers'].unique()}")
print(f"Lenders: {bis_long['lenders'].unique()}")

print("\nSample:")
print(bis_long.head(8).to_string(index=False))

# Save
bis_long.to_csv(OUT_DIR / "bis_credit_long.csv", index=False)
bis_long.to_parquet(OUT_DIR / "bis_credit_long.parquet", index=False)
meta_df.to_csv(OUT_DIR / "bis_series_metadata.csv", index=False)

print(f"\nSaved → {OUT_DIR / 'bis_credit_long.parquet'}")
print("Cell completed successfully.")

BIS Credit – Proper Parse
Total lines: 199
Detected series columns: 72

Series metadata sample:
 col_idx                             key country country_label borrowers   lenders valuation unit
       1 BIS:WS_TC(2.0):Q.AU.G.A.M.770.A      AU     Australia         G         A         M  770
       2 BIS:WS_TC(2.0):Q.AU.G.A.N.770.A      AU     Australia         G         A         N  770
       3 BIS:WS_TC(2.0):Q.AU.P.A.M.770.A      AU     Australia         P         A         M  770
       4 BIS:WS_TC(2.0):Q.AU.P.B.M.770.A      AU     Australia         P        "B         M  770
       5 BIS:WS_TC(2.0):Q.BE.G.A.M.770.A      BE       Belgium         G domestic"         M  770
       6 BIS:WS_TC(2.0):Q.BE.G.A.N.770.A      BE       Belgium         G         A         N  770
       7 BIS:WS_TC(2.0):Q.BE.P.A.M.770.A      BE       Belgium         P         A         M  770
       8 BIS:WS_TC(2.0):Q.BE.P.B.M.770.A      BE       Belgium         P         A         M  770

Countries in file: ['

In [5]:
# Inspect OECD ICIO structure (one year sample + file list)

from pathlib import Path
import pandas as pd

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
ICIO_DIR = PROJECT / "data" / "raw" / "oecd_icio"

print("=" * 60)
print("OECD ICIO – Structure Inspection")
print("=" * 60)

files = sorted(ICIO_DIR.glob("*_SML.csv"))
print(f"Found {len(files)} ICIO yearly files")
print(f"First: {files[0].name}")
print(f"Last : {files[-1].name}")

# Inspect one recent year (2020 if exists, else last)
sample_file = None
for y in [2020, 2018, 2015, 2022]:
    cand = ICIO_DIR / f"{y}_SML.csv"
    if cand.exists():
        sample_file = cand
        break
if sample_file is None:
    sample_file = files[-1]

print(f"\nInspecting: {sample_file.name}")
print(f"Size: {sample_file.stat().st_size / 1e6:.2f} MB")

# Read first few rows without assuming header
print("\n--- First 8 raw lines ---")
with open(sample_file, "r", encoding="utf-8", errors="replace") as f:
    for i, line in enumerate(f):
        if i >= 8:
            break
        print(f"{i:02d}| {line.rstrip()[:180]}")

# Try loading as dataframe
df = pd.read_csv(sample_file, low_memory=False)
print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns (first 20): {list(df.columns[:20])}")
print(f"Columns (last 10): {list(df.columns[-10:])}")

print("\n--- Head ---")
print(df.head(3).to_string())

print("\n--- First column sample values ---")
first_col = df.columns[0]
print(df[first_col].head(15).tolist())

print("\nCell completed.")

OECD ICIO – Structure Inspection
Found 20 ICIO yearly files
First: 1995_SML.csv
Last : 2017_SML.csv

Inspecting: 2017_SML.csv
Size: 87.44 MB

--- First 8 raw lines ---
00| V1,AGO_A01,AGO_A02,AGO_A03,AGO_B05,AGO_B06,AGO_B07,AGO_B08,AGO_B09,AGO_C10T12,AGO_C13T15,AGO_C16,AGO_C17_18,AGO_C19,AGO_C20,AGO_C21,AGO_C22,AGO_C23,AGO_C24A,AGO_C24B,AGO_C25,AGO_C2
01| AGO_A01,1946.7734,20.9635,36.7116,0.0019,0.1783,0.0011,0.2027,0.032,1008.0536,40.4125,0.1176,0.3069,0.0027,1.9315,3.253,2.6506,0.002,0.0016,5e-04,0.0263,0.0843,0.0389,0.153,0.2626,
02| AGO_A02,22.6272,135.4595,1.6957,0.0071,0.2424,0.0023,0.5256,0.0413,4.0834,0.7427,13.4464,4.3446,0.026,0.2128,0.1791,0.609,0.0164,0.0068,0.0018,0.0273,0.0292,0.0234,0.1009,0.108,0.0
03| AGO_A03,4.0041,0.244,241.7146,2e-04,0.0101,1e-04,0.031,0.0019,94.7746,0.17,0.004,0.0054,5e-04,0.0394,0.0649,0.0079,3e-04,2e-04,0,0.0018,0.0041,0.0041,0.0121,0.0136,7e-04,4e-04,0.11
04| AGO_B05,1.1178,0.0258,0.0309,0.0609,0.9888,0.0051,0.6562,0.3109,0.3212,0.1243,0.0057,0.0